# Validação do Ensemble com Especialista Local no CausalTime Traffic

Este notebook avalia em cinco trajetórias novas um ensemble com especialista local por aresta. O ranking combina 60% da melhor evidência local e 40% do consenso ponderado. Cada estratégia recebe exatamente a mesma trajetória, os mesmos dez nós e o mesmo pré-processamento.

A métrica principal é **Average Precision (AP)**. F1, precision, recall, SHD, AUROC, taxa de vitórias e tempo são resultados secundários. O ensemble é comparado com cada algoritmo que o compõe. Com cinco trajetórias, a análise continua exploratória: o menor p-valor bilateral possível do Wilcoxon é `0,0625`.

> Limite da conclusão: cinco trajetórias podem sugerir vantagem e estimar custo, mas não demonstram superioridade estatística no nível de 5%. As trajetórias também compartilham o mesmo grafo, portanto não sustentam generalização universal.

## 1. Protocolo pré-registrado

O critério confirmatório é mantido documentado para uma execução futura com mais trajetórias. O ensemble seria considerado superior a cada algoritmo avulso na métrica principal somente se, simultaneamente:

1. o limite inferior do IC 95% do ganho médio de AP for maior que `0,02`;
2. o p-valor pareado de Wilcoxon, corrigido por Holm, for menor que `0,05`;
3. o ensemble vencer em pelo menos 70% das trajetórias pareadas.

A configuração usa `PCMCI + DYNOTEARS + NeuralGrangercMLP + VARLiNGAM`, oito bootstraps em blocos, limiar binário `0,50` e peso local `0,60`. A evidência local combina força normalizada, estabilidade da aresta e confiabilidade do método, sem consultar o grafo verdadeiro.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import time

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from causal_discovery import (
    CausalPreprocessor,
    build_complete_undirected_pair_scores,
    compute_paired_superiority_statistics,
    compute_ranked_undirected_skeleton_metrics,
    compute_undirected_skeleton_metrics,
    get_registered_method_kwargs,
    get_registered_method_weights,
    get_registered_methods,
    load_time_series_dataset,
)
import causal_discovery.ensemble_selection as ensemble_selection_module

# Recarrega o módulo quando o framework foi alterado no mesmo kernel Jupyter.
ensemble_selection_module = importlib.reload(ensemble_selection_module)
select_robust_ensemble_combination = (
    ensemble_selection_module.select_robust_ensemble_combination
)

DATA_PATH = Path("datasets/causaltime/traffic/gen_data.npy")
GRAPH_PATH = Path("datasets/causaltime/traffic/graph.npy")
RESULTS_DIR = Path(".local/results/traffic_validation_local_expert_10nodes_seed42_5traj")

N_NODES = 10
NODE_SAMPLE_SEED = 42
N_TRAJECTORIES = 5
TRAJECTORY_SAMPLE_SEED = 2026
CALIBRATION_TRAJECTORIES = [42, 207, 210, 312, 369]
ENSEMBLE_METHOD_NAMES = [
    "PCMCI", "DYNOTEARS", "NeuralGrangercMLP", "VARLiNGAM",
]
COMPARATOR_METHOD_NAMES = []
MAX_LAG = 2
ENSEMBLE_THRESHOLD = 0.50
N_BOOTSTRAP = 8
LOCAL_EXPERT_WEIGHT = 0.60
RANDOM_STATE = 42
MEASURE_INDIVIDUAL_RUNTIMES = False

MINIMUM_AP_GAIN = 0.02
MINIMUM_WIN_RATE = 0.70
SIGNIFICANCE_LEVEL = 0.05
MIN_CONFIRMATORY_TRAJECTORIES = 10
STATISTICAL_BOOTSTRAPS = 10_000

OVERWRITE_CHECKPOINT = False
RETRY_FAILURES = True
STOP_ON_ERROR = False


## 2. Seleção de nós e amostra de trajetórias

São mantidos os mesmos dez nós sorteados por seed na calibração. As cinco trajetórias usadas para escolher a configuração são removidas do universo de amostragem; as trajetórias de validação são então sorteadas com seed `2026`. O `graph.npy` não participa do sorteio nem da seleção do ensemble.

In [ ]:
available_bundle = load_time_series_dataset(
    DATA_PATH,
    data_format="causaltime",
    graph_path=GRAPH_PATH,
    selected_columns=None,
    trajectory_index=0,
    column_prefix="traffic",
)
node_rng = np.random.default_rng(NODE_SAMPLE_SEED)
selected_nodes = sorted(node_rng.choice(
    available_bundle.available_columns,
    size=min(N_NODES, len(available_bundle.available_columns)),
    replace=False,
).tolist())

dataset_bundle = load_time_series_dataset(
    DATA_PATH,
    data_format="causaltime",
    graph_path=GRAPH_PATH,
    selected_columns=selected_nodes,
    trajectory_index=0,
    column_prefix="traffic",
)
nodes = list(dataset_bundle.selected_columns)
ground_truth = dataset_bundle.ground_truth.copy()

available_trajectory_indices = np.setdiff1d(
    np.arange(dataset_bundle.trajectory_count),
    np.asarray(CALIBRATION_TRAJECTORIES, dtype=int),
)
trajectory_rng = np.random.default_rng(TRAJECTORY_SAMPLE_SEED)
trajectory_indices = sorted(trajectory_rng.choice(
    available_trajectory_indices,
    size=min(N_TRAJECTORIES, dataset_bundle.trajectory_count),
    replace=False,
).tolist())

truth_summary = compute_undirected_skeleton_metrics(
    pd.DataFrame(columns=["source", "target", "lag"]),
    ground_truth,
    nodes=nodes,
)
print(f"Nós: {len(nodes)}")
print(f"Nós selecionados por seed {NODE_SAMPLE_SEED}: {nodes}")
print(f"Features temporais máximas no GES: {len(nodes) * (MAX_LAG + 1)}")
print(f"Pares possíveis: {truth_summary['candidate_pairs']}")
print(f"Pares verdadeiros: {truth_summary['ground_truth_pairs']}")
print(f"Prevalência: {truth_summary['ground_truth_prevalence']:.2%}")
print(f"Trajetórias de calibração excluídas: {CALIBRATION_TRAJECTORIES}")
print(f"Trajetórias selecionadas ({len(trajectory_indices)}): {trajectory_indices}")


## 3. Fun??es do experimento

Para o ranking, o ensemble usa `edge_probability`; m?todos com p-valor usam `1 - p_value` e os demais usam o m?dulo do score. A escala absoluta n?o interfere em AP/AUROC, que dependem da ordena??o dentro de cada estrat?gia. Pares n?o retornados recebem score zero. A pondera??o adaptativa usa somente as sa?das-base e os bootstraps, sem consultar o ground truth.

In [ ]:
def preprocess_trajectory(trajectory_index):
    raw = dataset_bundle.trajectory_frame(int(trajectory_index))
    preprocessor = CausalPreprocessor(
        raw, significance_level=0.05, decomposition_period=None
    )
    return preprocessor.fit_transform(
        make_stationary=True,
        normalize=True,
        remove_trend=False,
        max_diffs=2,
    )


def restrict_method_relations(method, allowed_relations):
    allowed_relations = set(allowed_relations)

    def run_restricted(data, **kwargs):
        result = method(data, **kwargs)
        if result is None or result.empty:
            return result
        mask = [
            (source, target) in allowed_relations
            for source, target in zip(result["source"], result["target"])
        ]
        return result.loc[mask].reset_index(drop=True)

    return run_restricted


def selection_arguments(processed_data):
    return {
        "min_methods": 4,
        "max_methods": 4,
        "min_votes": 1,
        "n_bootstrap": N_BOOTSTRAP,
        "block_size": max(2, len(processed_data) // 12),
        "stability_threshold": 0.60,
        "selection_probability_threshold": 0.50,
        "prior_edge_probability": 0.10,
        "posterior_weight": 0.70,
        "adaptive_method_weights": True,
        "stability_weight": 0.65,
        "local_expert_weight": LOCAL_EXPERT_WEIGHT,
        "method_stability_power": 1.0,
        "method_diversity_bonus": 0.15,
        "method_density_penalty": 0.50,
        "minimum_method_weight": 0.05,
        "confidence_level": 0.95,
        "random_state": RANDOM_STATE,
        "precompute_runs": True,
        "parallel_jobs": max(1, min(4, (os.cpu_count() or 2) - 1)),
        "max_bootstrap_seconds": 900,
    }


def evidence_mode(frame):
    if "p_value" in frame.columns:
        p_values = pd.to_numeric(frame["p_value"], errors="coerce")
        if np.isfinite(p_values).any():
            return "one_minus_p_value"
    return "absolute_score"


def evaluate_strategy(frame, strategy, trajectory_index, runtime_seconds, *, probability=False):
    binary = compute_undirected_skeleton_metrics(
        frame,
        ground_truth,
        prob_threshold=ENSEMBLE_THRESHOLD,
        nodes=nodes,
    )
    pair_scores = build_complete_undirected_pair_scores(
        frame,
        nodes,
        evidence=(
            "ensemble_score"
            if probability and "ensemble_score" in frame.columns
            else "probability" if probability
            else evidence_mode(frame)
        ),
    )
    ranked = compute_ranked_undirected_skeleton_metrics(pair_scores, ground_truth)
    return {
        "trajectory_index": int(trajectory_index),
        "strategy": str(strategy),
        "precision": binary["precision"],
        "recall": binary["recall"],
        "f1_score": binary["f1_score"],
        "structural_hamming_distance": binary["structural_hamming_distance"],
        "true_positives": binary["true_positives"],
        "false_positives": binary["false_positives"],
        "false_negatives": binary["false_negatives"],
        "average_precision": ranked["average_precision"],
        "roc_auc": ranked["roc_auc"],
        "runtime_seconds": float(runtime_seconds),
    }


def baseline_rows(trajectory_index):
    pairs = [(nodes[i], nodes[j]) for i in range(len(nodes)) for j in range(i + 1, len(nodes))]
    all_pairs = pd.DataFrame([
        {"source": source, "target": target, "lag": 1, "score": 1.0, "p_value": np.nan}
        for source, target in pairs
    ])
    rng = np.random.default_rng(RANDOM_STATE + int(trajectory_index))
    random_scores = rng.random(len(pairs))
    true_pair_count = truth_summary["ground_truth_pairs"]
    selected = np.argsort(random_scores)[-true_pair_count:]
    random_edges = pd.DataFrame([
        {"source": pairs[index][0], "target": pairs[index][1], "lag": 1,
         "score": random_scores[index], "p_value": np.nan}
        for index in selected
    ])
    random_pair_scores = pd.DataFrame([
        {"source": source, "target": target, "score": score}
        for (source, target), score in zip(pairs, random_scores)
    ])

    all_metrics = evaluate_strategy(all_pairs, "ALL_PAIRS", trajectory_index, 0.0)
    random_binary = compute_undirected_skeleton_metrics(random_edges, ground_truth, nodes=nodes)
    random_ranked = compute_ranked_undirected_skeleton_metrics(random_pair_scores, ground_truth)
    random_metrics = {
        "trajectory_index": int(trajectory_index), "strategy": "RANDOM_DENSITY",
        "precision": random_binary["precision"], "recall": random_binary["recall"],
        "f1_score": random_binary["f1_score"],
        "structural_hamming_distance": random_binary["structural_hamming_distance"],
        "true_positives": random_binary["true_positives"],
        "false_positives": random_binary["false_positives"],
        "false_negatives": random_binary["false_negatives"],
        "average_precision": random_ranked["average_precision"],
        "roc_auc": random_ranked["roc_auc"], "runtime_seconds": 0.0,
    }
    return [all_metrics, random_metrics]


def run_trajectory(
    trajectory_index, methods, comparator_methods, method_kwargs, method_weights
):
    processed = preprocess_trajectory(trajectory_index)
    relations = {(source, target) for source in nodes for target in nodes if source != target}
    restricted = {
        name: restrict_method_relations(method, relations)
        for name, method in methods.items()
    }
    restricted_comparators = {
        name: restrict_method_relations(method, relations)
        for name, method in comparator_methods.items()
    }

    individual_outputs = {}
    individual_runtimes = {
        name: np.nan for name in [*restricted, *restricted_comparators]
    }
    if MEASURE_INDIVIDUAL_RUNTIMES:
        for name, method in restricted.items():
            started = time.perf_counter()
            individual_outputs[name] = method(processed, **method_kwargs[name])
            individual_runtimes[name] = time.perf_counter() - started

    started = time.perf_counter()
    selection = select_robust_ensemble_combination(
        processed,
        restricted,
        method_kwargs=method_kwargs,
        method_weights=method_weights,
        expert_knowledge=[],
        **selection_arguments(processed),
    )
    ensemble_runtime = time.perf_counter() - started
    if not individual_outputs:
        for evaluation in selection["all_evaluations"].values():
            for name, output in evaluation["outputs"].items():
                individual_outputs.setdefault(name, output)
    for name, method in restricted_comparators.items():
        comparator_started = time.perf_counter()
        individual_outputs[name] = method(processed, **method_kwargs[name])
        individual_runtimes[name] = time.perf_counter() - comparator_started

    expected_outputs = set(restricted) | set(restricted_comparators)
    missing_outputs = sorted(expected_outputs - set(individual_outputs))
    if missing_outputs:
        raise RuntimeError(f"Saídas-base ausentes para: {missing_outputs}")

    rows = [
        evaluate_strategy(
            individual_outputs[name], name, trajectory_index, individual_runtimes[name]
        )
        for name in [*restricted, *restricted_comparators]
    ]
    summary = selection["best_evaluation"]["probabilistic_summary"].copy()
    rows.append(evaluate_strategy(
        summary, "ENSEMBLE", trajectory_index, ensemble_runtime, probability=True
    ))
    rows.extend(baseline_rows(trajectory_index))
    selection_row = {
        "trajectory_index": int(trajectory_index),
        "best_combination": " + ".join(selection["best_combination"]),
        "ensemble_runtime_seconds": ensemble_runtime,
        "processed_rows": len(processed),
        "adaptive_weights": json.dumps(
            selection["best_evaluation"]["effective_method_weights"],
            ensure_ascii=False, sort_keys=True,
        ),
        "dominant_method_counts": json.dumps(
            summary["dominant_method"].dropna().value_counts().to_dict(),
            ensure_ascii=False, sort_keys=True,
        ),
    }
    return rows, selection_row


## 4. Execução com checkpoint

A validação usa cinco trajetórias novas e uma única combinação de quatro métodos. Os resultados são salvos em `.local/results/traffic_validation_local_expert_10nodes_seed42_5traj`; o checkpoint anterior não será reutilizado porque pertence a outra regra de agregação.

In [ ]:
all_registered_methods = get_registered_methods()
all_method_kwargs = get_registered_method_kwargs(MAX_LAG)
all_method_weights = get_registered_method_weights()
methods = {name: all_registered_methods[name] for name in ENSEMBLE_METHOD_NAMES}
comparator_methods = {
    name: all_registered_methods[name] for name in COMPARATOR_METHOD_NAMES
}
method_kwargs = {
    name: all_method_kwargs[name]
    for name in [*ENSEMBLE_METHOD_NAMES, *COMPARATOR_METHOD_NAMES]
}
method_weights = {name: all_method_weights[name] for name in ENSEMBLE_METHOD_NAMES}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_path = RESULTS_DIR / "metrics.csv"
selections_path = RESULTS_DIR / "selections.csv"
failures_path = RESULTS_DIR / "failures.csv"
metadata_path = RESULTS_DIR / "configuration.json"

configuration = {
    "data_path": str(DATA_PATH), "graph_path": str(GRAPH_PATH),
    "nodes": nodes, "node_sample_seed": NODE_SAMPLE_SEED,
    "trajectory_indices": trajectory_indices,
    "max_lag": MAX_LAG, "n_bootstrap": N_BOOTSTRAP,
    "ensemble_threshold": ENSEMBLE_THRESHOLD,
    "local_expert_weight": LOCAL_EXPERT_WEIGHT,
    "ensemble_methods": list(methods),
    "comparator_methods": list(comparator_methods),
    "random_state": RANDOM_STATE,
    "measure_individual_runtimes": MEASURE_INDIVIDUAL_RUNTIMES,
    "minimum_ap_gain": MINIMUM_AP_GAIN,
    "minimum_win_rate": MINIMUM_WIN_RATE,
    "significance_level": SIGNIFICANCE_LEVEL,
    "min_confirmatory_trajectories": MIN_CONFIRMATORY_TRAJECTORIES,
}
if metadata_path.exists() and not OVERWRITE_CHECKPOINT:
    previous_configuration = json.loads(metadata_path.read_text(encoding="utf-8"))
    if previous_configuration != configuration:
        raise RuntimeError(
            "O checkpoint pertence a outra configuração. Altere RESULTS_DIR "
            "ou use OVERWRITE_CHECKPOINT=True conscientemente."
        )
if OVERWRITE_CHECKPOINT:
    for path in [metrics_path, selections_path, failures_path, metadata_path]:
        if path.exists():
            path.unlink()
metadata_path.write_text(
    json.dumps(configuration, indent=2, ensure_ascii=False), encoding="utf-8"
)

metrics_results = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
selection_results = pd.read_csv(selections_path) if selections_path.exists() else pd.DataFrame()
failure_results = pd.read_csv(failures_path) if failures_path.exists() else pd.DataFrame()
completed = set(
    metrics_results.loc[metrics_results.get("strategy", pd.Series(dtype=str)).eq("ENSEMBLE"),
                        "trajectory_index"].astype(int)
) if not metrics_results.empty else set()
failed = set(failure_results["trajectory_index"].astype(int)) if not failure_results.empty else set()

for position, trajectory_index in enumerate(trajectory_indices, start=1):
    if trajectory_index in completed or (trajectory_index in failed and not RETRY_FAILURES):
        print(f"[{position}/{len(trajectory_indices)}] trajetória {trajectory_index}: checkpoint")
        continue
    print(f"[{position}/{len(trajectory_indices)}] trajetória {trajectory_index}: executando...")
    started = time.perf_counter()
    try:
        rows, selection_row = run_trajectory(
            trajectory_index, methods, comparator_methods, method_kwargs, method_weights
        )
        metrics_results = pd.concat([metrics_results, pd.DataFrame(rows)], ignore_index=True)
        selection_results = pd.concat(
            [selection_results, pd.DataFrame([selection_row])], ignore_index=True
        )
        metrics_results.to_csv(metrics_path, index=False)
        selection_results.to_csv(selections_path, index=False)
        if not failure_results.empty:
            failure_results = failure_results.loc[
                failure_results["trajectory_index"].astype(int).ne(int(trajectory_index))
            ].reset_index(drop=True)
            if failure_results.empty:
                failures_path.unlink(missing_ok=True)
            else:
                failure_results.to_csv(failures_path, index=False)
        print(f"  concluída em {(time.perf_counter() - started) / 60:.1f} min")
    except Exception as error:
        failure_row = {
            "trajectory_index": int(trajectory_index),
            "error_type": type(error).__name__, "error": str(error),
        }
        failure_results = pd.concat(
            [failure_results, pd.DataFrame([failure_row])], ignore_index=True
        )
        failure_results.to_csv(failures_path, index=False)
        print(f"  FALHA: {type(error).__name__}: {error}")
        if STOP_ON_ERROR:
            raise

completed_count = (
    metrics_results.loc[metrics_results["strategy"].eq("ENSEMBLE"), "trajectory_index"].nunique()
    if not metrics_results.empty else 0
)
print(f"Trajetórias completas: {completed_count}")
print(f"Falhas registradas: {len(failure_results)}")


## 5. Resultados descritivos

A tabela resume desempenho e custo. `ALL_PAIRS` e `RANDOM_DENSITY` são controles: o primeiro prevê todos os pares; o segundo conhece apenas a densidade verdadeira, não quais pares são verdadeiros.

In [ ]:
if metrics_results.empty:
    if not failure_results.empty:
        display(failure_results)
        raise RuntimeError(
            "Nenhum resultado disponível. As causas reais estão na tabela acima. "
            "Reexecute as seções 1 a 4; RETRY_FAILURES=True refará as trajetórias."
        )
    raise RuntimeError("Nenhum resultado disponível. Execute primeiro a seção 4.")

summary_columns = [
    "average_precision", "roc_auc", "precision", "recall",
    "f1_score", "structural_hamming_distance", "runtime_seconds",
]
descriptive = metrics_results.groupby("strategy")[summary_columns].agg(["mean", "std", "median"])
display(descriptive.round(3))

combination_frequency = (
    selection_results["best_combination"].value_counts().rename_axis("combination")
    .reset_index(name="trajectories")
) if not selection_results.empty else pd.DataFrame()
display(combination_frequency)

if not selection_results.empty and "adaptive_weights" in selection_results:
    adaptive_weight_history = pd.json_normalize(
        selection_results["adaptive_weights"].map(json.loads)
    )
    adaptive_weight_history.index = selection_results["trajectory_index"].astype(int)
    adaptive_weight_history.index.name = "trajectory_index"
    print("Pesos adaptativos por trajetória:")
    display(adaptive_weight_history.round(3))

if not selection_results.empty and "dominant_method_counts" in selection_results:
    dominant_method_history = pd.json_normalize(
        selection_results["dominant_method_counts"].map(json.loads)
    ).fillna(0).astype(int)
    dominant_method_history.index = selection_results["trajectory_index"].astype(int)
    dominant_method_history.index.name = "trajectory_index"
    print("Quantidade de arestas em que cada método foi o especialista local:")
    display(dominant_method_history)

figure = px.box(
    metrics_results, x="strategy", y="average_precision", points="all",
    title="Average Precision por estratégia e trajetória",
)
figure.update_xaxes(tickangle=45)
figure.show()


## 6. Análise pareada exploratória

O ensemble é comparado ao PCMCI e aos três componentes congelados. A correção de Holm é calculada, mas cinco trajetórias não fornecem resolução suficiente para significância bilateral de 5%. O resultado principal é a comparação Ensemble versus PCMCI nas trajetórias de validação.

In [ ]:
def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    order = np.argsort(values)
    adjusted_sorted = np.maximum.accumulate(
        np.array([(len(values) - rank) * values[index] for rank, index in enumerate(order)])
    )
    adjusted = np.empty_like(values)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted


individual_methods = [*methods, *comparator_methods]
primary_rows = [
    compute_paired_superiority_statistics(
        metrics_results, candidate="ENSEMBLE", baseline=baseline,
        metric="average_precision", higher_is_better=True,
        n_bootstrap=STATISTICAL_BOOTSTRAPS, random_state=RANDOM_STATE,
    )
    for baseline in individual_methods
]
primary_comparison = pd.DataFrame(primary_rows)
primary_comparison["holm_p_value"] = holm_adjust(primary_comparison["wilcoxon_p_value"])
primary_comparison["confirmatory_sample_available"] = (
    primary_comparison["paired_trajectories"] >= MIN_CONFIRMATORY_TRAJECTORIES
)
primary_comparison["superiority_criterion_met"] = (
    primary_comparison["confirmatory_sample_available"]
) & (
    primary_comparison["confidence_interval_low"] > MINIMUM_AP_GAIN
) & (
    primary_comparison["holm_p_value"] < SIGNIFICANCE_LEVEL
) & (
    primary_comparison["win_rate"] >= MINIMUM_WIN_RATE
)
display(primary_comparison.sort_values("baseline").round(4))

secondary_rows = []
for metric, higher_is_better in [("f1_score", True), ("structural_hamming_distance", False)]:
    for baseline in individual_methods:
        secondary_rows.append(compute_paired_superiority_statistics(
            metrics_results, candidate="ENSEMBLE", baseline=baseline,
            metric=metric, higher_is_better=higher_is_better,
            n_bootstrap=STATISTICAL_BOOTSTRAPS, random_state=RANDOM_STATE,
        ))
secondary_comparison = pd.DataFrame(secondary_rows)
display(secondary_comparison.round(4))


## 7. Conclusão automática do estudo piloto

A conclusão abaixo bloqueia afirmações confirmatórias quando há menos de dez trajetórias. Com cinco trajetórias, a redação permitida descreve apenas tendência, magnitude, consistência e custo observados.

In [ ]:
passed = primary_comparison.loc[primary_comparison["superiority_criterion_met"], "baseline"].tolist()
failed = primary_comparison.loc[~primary_comparison["superiority_criterion_met"], "baseline"].tolist()
confirmatory_available = bool(primary_comparison["confirmatory_sample_available"].all())

if not confirmatory_available:
    print(
        f"Estudo piloto: somente {int(primary_comparison['paired_trajectories'].min())} "
        f"trajetórias pareadas; o mínimo operacional definido é "
        f"{MIN_CONFIRMATORY_TRAJECTORIES}."
    )
    print(
        "Conclusão permitida: os resultados sugerem ou não sugerem vantagem do ensemble "
        "neste piloto. Reporte ganho médio, intervalo, taxa de vitórias e custo, sem "
        "afirmar superioridade estatisticamente demonstrada."
    )
elif len(passed) == len(individual_methods):
    print(
        "Conclusão permitida: no conjunto de trajetórias avaliado do CausalTime Traffic, "
        "o ensemble apresentou AP superior a todos os algoritmos avulsos registrados, "
        "segundo o critério pré-especificado."
    )
else:
    print(
        "Conclusão permitida: não foi demonstrada superioridade do ensemble sobre todos "
        "os algoritmos avulsos segundo o critério pré-especificado. Reporte quais "
        "comparações foram positivas e quais permaneceram inconclusivas."
    )
print("Comparações sem critério confirmatório atendido:", failed)
print(
    "Não concluir superioridade universal sem repetir o protocolo em datasets com grafos diferentes."
)
